# 07 — Prédiction de liens manquants (Link Prediction)

**Objectif :** parmi toutes les paires de cytokines qui n'ont **pas** d'interaction connue
dans STRING, on essaie de prédire lesquelles ont le plus de chances d'interagir quand même,
en se basant uniquement sur la structure du réseau existant (pas sur de nouvelles données
biologiques). C'est une vraie composante de machine learning sur graphe, qui différencie
ton projet d'une simple analyse descriptive.

**Principe intuitif :** si deux cytokines A et B ne sont pas connectées, mais qu'elles
partagent beaucoup de "voisins communs" (d'autres cytokines connectées aux deux à la fois),
il est statistiquement plus probable qu'elles interagissent aussi — comme deux personnes qui
ont beaucoup d'amis en commun ont plus de chances de se connaître elles-mêmes.

**Données utilisées :** `data/raw/string_network.csv`

In [1]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

network_df = pd.read_csv("../data/raw/string_network.csv")
G = nx.from_pandas_edgelist(
    network_df, "preferredName_A", "preferredName_B", edge_attr="score"
)
print(f"Graphe chargé : {G.number_of_nodes()} nœuds, {G.number_of_edges()} arêtes")

n_possible_pairs = G.number_of_nodes() * (G.number_of_nodes() - 1) // 2
n_missing_pairs = n_possible_pairs - G.number_of_edges()
print(f"Paires sans interaction connue à prédire : {n_missing_pairs}")

Graphe chargé : 21 nœuds, 147 arêtes
Paires sans interaction connue à prédire : 63


## 1. Trois méthodes classiques de prédiction de liens

- **Common Neighbors** : nombre brut de voisins partagés entre A et B
- **Jaccard Coefficient** : proportion de voisins communs par rapport à l'ensemble des voisins des deux (normalisé, entre 0 et 1)
- **Adamic-Adar Index** : comme Common Neighbors, mais donne plus de poids aux voisins communs qui sont eux-mêmes "rares"/peu connectés (un voisin commun peu connecté est un indice plus fort qu'un voisin commun connecté à tout le monde)

In [2]:
# networkx calcule ces scores uniquement pour les paires de nœuds NON connectées par défaut
jaccard_scores = list(nx.jaccard_coefficient(G))
adamic_adar_scores = list(nx.adamic_adar_index(G))

# Common neighbors n'a pas de fonction dédiée pour toutes les paires -> calcul manuel
common_neighbors_scores = []
non_edges = list(nx.non_edges(G))
for u, v in non_edges:
    cn = len(list(nx.common_neighbors(G, u, v)))
    common_neighbors_scores.append((u, v, cn))

# Fusion des 3 méthodes dans un seul tableau
jaccard_df = pd.DataFrame(jaccard_scores, columns=["cytokine_A", "cytokine_B", "jaccard"])
adamic_df = pd.DataFrame(adamic_adar_scores, columns=["cytokine_A", "cytokine_B", "adamic_adar"])
common_df = pd.DataFrame(common_neighbors_scores, columns=["cytokine_A", "cytokine_B", "common_neighbors"])

predictions_df = jaccard_df.merge(adamic_df, on=["cytokine_A", "cytokine_B"])
predictions_df = predictions_df.merge(common_df, on=["cytokine_A", "cytokine_B"])

print(f"{len(predictions_df)} paires non connectées évaluées.")
predictions_df.head()

63 paires non connectées évaluées.


,cytokine_A,cytokine_B,jaccard,adamic_adar,common_neighbors
0,CXCL10,TGFB1,0.785714,3.856700,11
1,CXCL10,IL12A,0.000000,0.000000,0
2,CXCL10,IL23A,0.312500,1.730508,5
3,CXCL10,IL12B,0.187500,1.031576,3
4,CXCL10,IL1RN,0.571429,2.797094,8


## 2. Top 15 des interactions les plus probables (non encore documentées dans STRING)

In [3]:
# On combine les 3 scores en un score composite simple (moyenne des rangs)
# -> évite qu'une seule méthode ne domine artificiellement le classement
for col in ["jaccard", "adamic_adar", "common_neighbors"]:
    predictions_df[f"rank_{col}"] = predictions_df[col].rank(ascending=False)

predictions_df["composite_rank_score"] = predictions_df[
    ["rank_jaccard", "rank_adamic_adar", "rank_common_neighbors"]
].mean(axis=1)

top_predictions = predictions_df.sort_values("composite_rank_score").head(15)
top_predictions[["cytokine_A", "cytokine_B", "jaccard", "adamic_adar", "common_neighbors"]]

,cytokine_A,cytokine_B,jaccard,adamic_adar,common_neighbors
5,CXCL10,IL22,0.800000,4.234565,12
15,IL22,CCL11,0.800000,4.234565,12
18,IL22,CCL2,0.705882,4.234565,12
0,CXCL10,TGFB1,0.785714,3.856700,11
28,TGFB1,CCL11,0.785714,3.856700,11
29,TGFB1,IL5,0.733333,3.856700,11
20,IL9,TGFB1,0.733333,3.856700,11
14,IL22,TGFB1,0.714286,3.496027,10
33,TGFB1,IL1RN,0.727273,2.797094,8
4,CXCL10,IL1RN,0.571429,2.797094,8


In [4]:
top_predictions.to_csv("../data/processed/link_predictions_top15.csv", index=False)
print("Sauvegardé dans data/processed/link_predictions_top15.csv")

Sauvegardé dans data/processed/link_predictions_top15.csv


## 3. Validation simple de la méthode (auto-évaluation)

**Principe :** avant de faire confiance à ces prédictions, on vérifie que la méthode fonctionne
sur des données qu'on connaît déjà. On retire artificiellement 20% des interactions réelles
du graphe, puis on demande à la méthode de les "redécouvrir" parmi toutes les paires non
connectées. Si les arêtes retirées ressortent bien en haut du classement, la méthode est fiable.

In [5]:
import random
random.seed(42)

all_edges = list(G.edges())
n_test = int(0.2 * len(all_edges))
test_edges = random.sample(all_edges, n_test)

# Graphe "amputé" de ces arêtes de test
G_train = G.copy()
G_train.remove_edges_from(test_edges)

# On recalcule Adamic-Adar sur ce graphe amputé, pour TOUTES les paires non connectées
aa_train = {(u, v): score for u, v, score in nx.adamic_adar_index(G_train)}

# Rang des arêtes retirées (qu'on connaît) dans ce nouveau classement
all_scores_sorted = sorted(aa_train.items(), key=lambda x: x[1], reverse=True)
ranks_of_removed_edges = []
for i, ((u, v), score) in enumerate(all_scores_sorted):
    if (u, v) in test_edges or (v, u) in test_edges:
        ranks_of_removed_edges.append(i + 1)

print(f"{n_test} arêtes retirées pour le test.")
print(f"Rang moyen des vraies arêtes retrouvées : {sum(ranks_of_removed_edges)/len(ranks_of_removed_edges):.1f}")
print(f"(sur {len(all_scores_sorted)} paires non connectées au total -> plus le rang moyen est petit, mieux c'est)")

29 arêtes retirées pour le test.
Rang moyen des vraies arêtes retrouvées : 20.8
(sur 92 paires non connectées au total -> plus le rang moyen est petit, mieux c'est)


**Comment interpréter ce résultat :** si le rang moyen des arêtes retirées est nettement
meilleur (plus petit) que ce qu'on obtiendrait au hasard (qui serait environ la moitié du
nombre total de paires), cela confirme que la méthode Adamic-Adar capture un vrai signal
structurel — donc que les nouvelles prédictions du haut du classement (section 2) sont
probablement pertinentes, et pas du bruit.

## Résumé de ce notebook

- Calculé 3 méthodes de prédiction de liens (Common Neighbors, Jaccard, Adamic-Adar) sur
  toutes les paires de cytokines non connectées
- Produit un classement composite des 15 interactions les plus probables mais non encore
  documentées dans STRING (`data/processed/link_predictions_top15.csv`)
- Validé la fiabilité de la méthode en la testant sur des arêtes réelles retirées artificiellement

**Limite à mentionner honnêtement dans le README :** ces prédictions sont des hypothèses
statistiques basées uniquement sur la topologie du réseau — elles nécessiteraient une
validation expérimentale réelle (ou au moins une recherche bibliographique ciblée) avant
d'être considérées comme des découvertes biologiques.